# Load Pilot 1 CSV and post-processing
This notebook loads the dataset from `Pilot_1_full.csv` and shows a quick preview.

In [ ]:
import pandas as pd
from pathlib import Path
import json

csv_path = Path('Annotations_05_05_2026.csv')
df = pd.read_csv(csv_path)

# Remove rows without a data_json value
df = df.dropna(subset=['data_json'])
df = df[df['data_json'].astype(str).str.strip() != '']

print(f'Loaded {csv_path} successfully.')
print(f'Shape: {df.shape[0]} rows x {df.shape[1]} columns')
df.head()

In [ ]:
# Allow JSON-style literals in the next cell's pasted content
null = None
true = True
false = False

In [ ]:
# Add video_id with pattern by 6-row sets inside 36-row blocks BEFORE expansion:
# 1,2,3,4,5,6,1,2,3,4,5,6,1,2,3,4,5,6,1,2,3,4,5,6,1,2,3,4,5,6,1,2,3,4,5,6,
# 7,8,9,10,11,12,7,8,9,10,11,12,...
df_with_video_id = df.copy()
video_ids = []
for idx in range(len(df_with_video_id)):
    block_index = idx // 36
    offset_in_block = idx % 36
    video_id = block_index * 6 + (offset_in_block % 6) + 1
    video_ids.append(video_id)

df_with_video_id['video_id'] = video_ids

In [ ]:
def extract_last_objects_expanded(data_json_str):
    """Parse data_json and create a list of (mention_id, last_object) pairs for each key"""
    # Handle NaN or non-string values
    if pd.isna(data_json_str):
        return []
    
    # If already a dict, use as-is
    if isinstance(data_json_str, dict):
        data = data_json_str
    else:
        # Parse as JSON string
        try:
            data = json.loads(str(data_json_str))
        except (json.JSONDecodeError, ValueError):
            return []
    
    result = []
    
    for mention_id, objects_list in data.items():
        # Keep only the last object in the list for this key
        if objects_list:
            result.append((mention_id, objects_list[-1]))
    
    return result


# Extract and expand rows: one row per mention_id key
expanded_data = []
for idx, row in df_with_video_id.iterrows():
    mention_pairs = extract_last_objects_expanded(row['data_json'])
    for mention_id, last_obj in mention_pairs:
        new_row = row.copy()
        # Create a new data_json with only this mention_id and its last object
        new_row['data_json'] = json.dumps({mention_id: last_obj})
        expanded_data.append(new_row)

df_last_only = pd.DataFrame(expanded_data).reset_index(drop=True)

print("Added video_id pattern to original rows, then expanded data_json to one row per mention key.")
print(f"Original shape: {df.shape}")
print(f"Expanded shape: {df_last_only.shape}")
print("\nFirst 36 video_id values (should cycle 1-6):")
print(df_last_only['video_id'].head(36).tolist())

In [ ]:
# Remove unused metadata fields from all working dataframes
cols_to_remove = ['created_at', 'updated_at', 'interface', 'category']

for name, frame in [('df_last_only', df_last_only)]:
    existing = [c for c in cols_to_remove if c in frame.columns]
    frame.drop(columns=existing, inplace=True)
    print(f"{name}: removed {existing}")

print('\nRemaining columns in df_form_A:')
print(df_last_only.columns.tolist())

In [ ]:
# Add narrative fields from data_json into top-level dataframe columns
narrative_fields = [
    'intention_description',
    'intention_description_confidence',
    'intention_explanation',
    'intention_explanation_confidence',
    'intention_intensity',
    'counterfactual_explanation',
    'timestamp_start',
    'timestamp_end',
]


def extract_narrative_fields(data_json_value):
    if isinstance(data_json_value, str):
        data = json.loads(data_json_value)
    else:
        data = data_json_value

    # Each row keeps one object per mention after prior cleaning
    for obj in data.values():
        narratives = obj.get('narratives', {})
        return {field: narratives.get(field, None) for field in narrative_fields}

    return {field: None for field in narrative_fields}


def add_narrative_columns(frame):
    extracted = frame['data_json'].apply(extract_narrative_fields)
    extracted_df = pd.DataFrame(extracted.tolist(), index=frame.index)
    return pd.concat([frame, extracted_df], axis=1)


# Apply to all working dataframes
df_last_only = add_narrative_columns(df_last_only)


In [ ]:
# Interactive full-value table for narrative fields (Form A and B), left-aligned
from itables import init_notebook_mode, show

init_notebook_mode(all_interactive=True)

# Prepare a clean display dataframe to avoid DataTables parsing issues
table_df_last_only = df_last_only[["task_id", "participant", "video_id"] + narrative_fields].copy()
for col in table_df_last_only.columns:
    table_df_last_only[col] = table_df_last_only[col].fillna('').astype(str)

print('Displaying interactive table for Form Option A narrative fields:')
show(
    table_df_last_only,
    scrollX=True,
    scrollY='320px',
    paging=False,
    order=[],
    columnDefs=[{'targets': '_all', 'className': 'dt-head-left dt-body-left'}]
)


#### Process Model outputs

In [ ]:
from pathlib import Path
import json

response_files = [
    Path("model_responses/annotation_clips_0-250.json"),
    Path("model_responses/annotation_clips_251-500.json"),
    Path("model_responses/annotation_clips_501-700.json"),
]

response_frames = []

for response_file in response_files:
    with response_file.open("r", encoding="utf-8") as handle:
        payload = json.load(handle)

    results = payload.get("results", [])
    if not results:
        continue

    frame = pd.json_normalize(results)
    frame.insert(0, "source_file", response_file.name)

    summary = payload.get("__summary__", {})
    for key, value in summary.items():
        frame[f"summary_{key}"] = value

    response_frames.append(frame)

model_responses_df = pd.concat(response_frames, ignore_index=True) if response_frames else pd.DataFrame()

print(f"Loaded {len(response_frames)} files into model_responses_df")
print(f"Shape: {model_responses_df.shape[0]} rows x {model_responses_df.shape[1]} columns")
print("Columns:", model_responses_df.columns.tolist())

In [ ]:
existing_drop_cols = [
    'source_file',
    'source_video_path',
    'rewritten_video_path',
    'participant_image_path',
    'source_audio_paths',
    'rewritten_audio_paths',
    'conversation_floor_audio_paths',
    'summary_input_json',
    'summary_prompt_config',
    'summary_record_count',
    'summary_selected_record_count',
    'summary_start_index',
    'summary_end_index',
    'summary_retained_count',
    'summary_skipped_count',
    'summary_processed_count',
    'summary_error_count',
    'summary_no_audio',
    'summary_max_video_frames',
    'summary_aggregated_audio_dir',
    'summary_skip_reasons',
]
model_responses_df = model_responses_df.drop(columns=[c for c in existing_drop_cols if c in model_responses_df.columns])
metadata_cols = ['video_path', 'audio_paths', 'participant_audio_path', 'aggregated_conversation_floor_audio_path', 'audio_warnings', 'system', 'user']
model_responses_df['metadata'] = model_responses_df.apply(lambda row: row.reindex(metadata_cols).to_dict(), axis=1)
model_responses_df = model_responses_df.drop(columns=metadata_cols, errors='ignore')
model_responses_df.head()
print("Columns:", model_responses_df.columns.tolist())
print(f"Shape: {model_responses_df.shape[0]} rows x {model_responses_df.shape[1]} columns")



In [ ]:
import re
import json


def parse_assistant_response(text):
    if pd.isna(text):
        return []

    normalized = str(text).strip()
    if not normalized:
        return []

    pattern = re.compile(r'(?ms)^\s*\d+\.\s*(Timestamps|Intention|Confidence in interpretation|Why|Confidence in explanation|Intensity|Counterfactual explanation):\s*(.*?)(?=^\s*\d+\.\s*(?:Timestamps|Intention|Confidence in interpretation|Why|Confidence in explanation|Intensity|Counterfactual explanation):\s*|\Z)')
    matches = list(pattern.finditer(normalized))

    if not matches:
        lines = [line.strip() for line in normalized.splitlines() if line.strip()]
        if len(lines) >= 2 and lines[0].lower().startswith('no clear intention'):
            return [{
                'timestamps': None,
                'intention_description': None,
                'intention_description_confidence': None,
                'intention_explanation': lines[1].split(':', 1)[1].strip() if ':' in lines[1] else lines[1],
                'intention_explanation_confidence': None,
                'intention_intensity': None,
                'counterfactual_explanation': None,
            }]
        return []

    blocks = []
    current = {}

    for match in matches:
        label = match.group(1).strip()
        value = match.group(2).strip()

        if label == 'Timestamps' and current:
            blocks.append(current)
            current = {}

        current[label] = value

    if current:
        blocks.append(current)

    def _clean(value):
        if value is None:
            return None
        cleaned = value.strip()
        return cleaned if cleaned else None

    def _to_int(value):
        cleaned = _clean(value)
        if cleaned is None:
            return None
        match = re.search(r'\d+', cleaned)
        return int(match.group()) if match else None

    parsed = []
    for block in blocks:
        parsed.append({
            'timestamps': _clean(block.get('Timestamps')),
            'intention_description': _clean(block.get('Intention')),
            'intention_description_confidence': _to_int(block.get('Confidence in interpretation')),
            'intention_explanation': _clean(block.get('Why')),
            'intention_explanation_confidence': _to_int(block.get('Confidence in explanation')),
            'intention_intensity': _to_int(block.get('Intensity')),
            'counterfactual_explanation': _clean(block.get('Counterfactual explanation')),
        })

    return parsed


model_responses_df['assistant_parsed'] = model_responses_df['assistant'].apply(parse_assistant_response)
print('Parsed assistant responses into assistant_parsed')
unparsed_mask = model_responses_df['assistant_parsed'].apply(len).eq(0)
unparsed_rows = model_responses_df.loc[unparsed_mask, ['record_id', 'assistant']].copy() if 'record_id' in model_responses_df.columns else model_responses_df.loc[unparsed_mask, ['assistant']].copy()
print(f'Unparsed rows: {len(unparsed_rows)}')
print(f'number of annotations: {model_responses_df.shape[0]}')
if len(unparsed_rows):
    for _, row in unparsed_rows.head(5).iterrows():
        print('---')
        if 'record_id' in unparsed_rows.columns:
            print(f"record_id: {row['record_id']}")
        print(row['assistant'])

In [ ]:
# Expand each parsed assistant JSON object into its own row
assistant_flat_rows = []

for _, row in model_responses_df.iterrows():
    parsed_items = row['assistant_parsed'] if isinstance(row['assistant_parsed'], list) else []
    if not parsed_items:
        base_row = row.drop(labels=['assistant_parsed']).to_dict()
        base_row['timestamps'] = None
        base_row['intention_description'] = None
        base_row['intention_description_confidence'] = None
        base_row['intention_explanation'] = None
        base_row['intention_explanation_confidence'] = None
        base_row['intention_intensity'] = None
        base_row['counterfactual_explanation'] = None
        assistant_flat_rows.append(base_row)
        continue

    for parsed_item in parsed_items:
        base_row = row.drop(labels=['assistant_parsed']).to_dict()
        base_row.update(parsed_item)
        assistant_flat_rows.append(base_row)

df_model = pd.DataFrame(assistant_flat_rows)
df_model.to_csv('df_model.csv', index=False)
print(f'Flattened assistant_parsed into {len(df_model)} rows')
print('Saved df_model.csv')
print('Columns:', df_model.columns.tolist())
df_model.head()

#### LLM Parse

In [ ]:
from pathlib import Path
from openai import OpenAI


def _parse_yes_no_to_bool(reply_text: str) -> bool:
    """Convert a model reply into a strict boolean (yes=True, no=False)."""
    normalized = (reply_text or "").strip().lower()
    if normalized.startswith("yes"):
        return True
    if normalized.startswith("no"):
        return False
    raise ValueError(f"Unexpected model response: {reply_text!r}")


def _emotion_mentioned(
    text: str,
    client: OpenAI,
    user_prompt_template: str,
    model: str = "chat",
) -> bool:
    """Return True/False by calling the OpenAI-compatible Tulip client.

    Calls `client.chat.completions.create(...)` and extracts the textual reply
    robustly from common response shapes.
    """
    if "{text}" in user_prompt_template:
        user_prompt = user_prompt_template.format(text=text)
    else:
        user_prompt = f"{user_prompt_template}\n\nText:\n{text}"

    system_msg = {"role": "system", "content": "You are a strict binary classifier. Answer with exactly one word: yes or no."}
    user_msg = {"role": "user", "content": user_prompt}

    response = client.chat.completions.create(model=model, messages=[system_msg, user_msg], temperature=0)

    # Extract answer text from several possible response shapes
    answer = ""
    try:
        if isinstance(response, dict):
            # dict-style response
            choices = response.get("choices") or []
            if choices:
                first = choices[0]
                if isinstance(first, dict):
                    msg = first.get("message") or first.get("delta")
                    if isinstance(msg, dict):
                        content = msg.get("content")
                        if isinstance(content, str):
                            answer = content
                    elif isinstance(first.get("text"), str):
                        answer = first.get("text")
        else:
            # SDK-style object
            choices = getattr(response, "choices", None)
            if choices:
                first = choices[0]
                msg = getattr(first, "message", None)
                if msg is not None:
                    content = getattr(msg, "content", None)
                    if isinstance(content, str):
                        answer = content
                text_attr = getattr(first, "text", None)
                if not answer and isinstance(text_attr, str):
                    answer = text_attr

        answer = (answer or "").strip()
    except Exception:
        answer = ""

    return _parse_yes_no_to_bool(answer)


def _read_api_key_from_file(file_name: str = "tulip.key") -> str:
    """Read Tulip API key from a local file in the current working directory."""
    key_path = Path(file_name)
    if not key_path.exists():
        raise FileNotFoundError(
            f"Could not find {file_name} in {Path.cwd()}. Create the file and put your API key in it."
        )

    api_key = key_path.read_text(encoding="utf-8").strip()
    if not api_key:
        raise ValueError(f"{file_name} is empty. Add your API key to this file.")

    return api_key


def add_emotion_flag(
    frame: pd.DataFrame,
    user_prompt_template: str = (
        "Does this text mention any reference to the emotion of the participant "
        "(for example: happy, sad, angry, afraid, anxious, hopeful, "
        "frustrated, proud, etc.)?\n\nText:\n{text}"
    ),
    description_col: str = "intention_description",
    explanation_col: str = "intention_explanation",
    output_col: str = "emotion_mentioned",
    model: str = "chat",
    base_url: str = "https://api.tulip.tudelft.nl/chat/v1/",
) -> pd.DataFrame:
    """
    Loop over every row, apply the prompt to intention_description +
    intention_explanation, and return a dataframe with a boolean output column.
    """
    api_key = _read_api_key_from_file("tulip.key")
    client = OpenAI(base_url=base_url, api_key=api_key)
    result = frame.copy()

    flags = []
    for _, row in result.iterrows():
        description = "" if pd.isna(row.get(description_col)) else str(row.get(description_col))
        explanation = "" if pd.isna(row.get(explanation_col)) else str(row.get(explanation_col))
        combined_text = f"{description}\n\n{explanation}".strip()

        if not combined_text:
            flags.append(False)
            continue

        try:
            flags.append(
                _emotion_mentioned(
                    combined_text,
                    client=client,
                    user_prompt_template=user_prompt_template,
                    model=model,
                )
            )
        except Exception:
            flags.append(False)

    result[output_col] = flags
    return result

In [ ]:
emotion_prompt = (
        "The following text is an annotation of a participant's intention in a social situation."
        "Does this text mention any reference to the emotion of the participant?"
        "\n\nText:\n{text}"
    )
emotion_prompt_v2 = (
    """
    Your task is to determine whether the given text contains ANY reference to the participant’s mental or emotional state.

    Definition:
    A reference to mental or emotional state includes:
    - Explicit mentions of emotions or feelings
    (e.g., "He is happy", "She feels nervous", "They are angry")
    - Mentions of intentions, desires, beliefs, or thoughts
    (e.g., "He wants to leave", "She is trying to impress him", "They think it is funny")
    - Interpretations of internal states based on behavior
    (e.g., "He seems uncomfortable", "She appears confused", "They are enjoying the moment")

    Do NOT count:
    - Pure descriptions of observable or audible behavior with no interpretation
    (e.g., "He crossed his arms", "She is talking to him", "They are sitting down")
    - References to actions alone, even if they could imply emotion, unless an internal state is explicitly or implicitly inferred

    If there is ANY mention or inference of a mental or emotional state, answer "yes".
    Otherwise, answer "no".

    Text:
    {text}
    """
    )

cues_prompt = (
        "The following text is an annotation of a participant's intention in a social situation."
        "Does this text mention any direct reference to the video/audio? Such as cues, gestures, things the participant said or did, or any direct reference to the video/audio content?"
        "\n\nText:\n{text}"
    )

cues_prompt_v2 = (
    """
    Your task is to determine whether the given text contains ANY reference to observable or audible cues from a video.

    Definition:
    A reference to video/audio cues includes:
    - Direct descriptions of observable actions, gestures, objects, sounds, or locations
    (e.g., "She pointed at him", "He is holding a phone", "A door slams")
    - Indirect references to such cues
    (e.g., "He answered her question" → implies she spoke,
            "They reacted to the noise" → implies a sound occurred)

    Do NOT count:
    - Pure interpretations, intentions, or mental states with NO reference to observable/audible evidence
    (e.g., "He feels nervous", "She wants to impress him")
    - General summaries that could be made without seeing or hearing the video

    If there is ANY mention or implication of observable or audible evidence, answer "yes".
    Otherwise, answer "no".

    Text:
    {text}
    """
    )

belief_prompt = (
        "The following text is an annotation of a participant's intention in a social situation."
        "Does this text mention any reference to the participant's beliefs, thoughts, or mental belief states (for example: what they think, believe, know, or assume)?"
        "\n\nText:\n{text}"
    )

belief_prompt_v2 = (
    """
    Your task is to determine whether the given text contains ANY reference to beliefs or thought states.

    Definition:
    A reference to belief includes:
    - Statements about what the participant thinks, believes, knows, or assumes
    (e.g., "He thinks she is upset", "She believes he is lying", "They know the answer")
    - Statements about what the annotator believes about the situation
    (e.g., "It seems like he misunderstood the situation", "She is probably unaware of what happened")
    - Second-order beliefs (beliefs about others' beliefs)
    (e.g., "He thinks she knows the truth", "She believes he doesn’t realize it")

    Do NOT count:
    - Pure emotional states (e.g., "He feels nervous", "She is happy")
    - Pure intentions or desires without belief/thought content
    (e.g., "He wants to leave", "She is trying to impress him")
    - Pure descriptions of observable behavior
    (e.g., "He is talking to her", "She looks away")

    If there is ANY mention or inference of a belief, thought, knowledge, or assumption, answer "yes".
    Otherwise, answer "no".

    Text:
    {text}
    """
)

intention_prompt = (
        "The following text is an annotation of a participant's intention in a social situation."
        "Does this text describe the intention of the participant? If there is a description of the participant's intention, answer yes. If there is no description of the participant's intention, just a general description of the particiapant's action, what they are doing or things happening in the scene, answer no."
        "An intention is what a person wants to achieve, based on beliefs about the situation and what they desire to happen."
        "\n\nText:\n{text}"
    )

# Note: I do not belive this is enough to classify the it as an intention annotation (it is more about the goal or desire)
# But if an annotation contains this plus any one of the emotion, belief, or cues references, then it is more likely to be an intention annotation.
intention_prompt_v2 = (
    """
    Your task is to determine whether the given text contains ANY description of the participant’s intention.

    Definition:
    An intention is a goal-directed mental state describing what the participant is trying to achieve or bring about. 
    It typically explains WHY a behavior is performed, not just WHAT is happening.

    A reference to intention includes:
    - Explicit statements of goals or desired outcomes
    (e.g., "He wants to leave", "She is trying to impress him", "They intend to start a conversation")
    - Descriptions of actions linked to a purpose or goal
    (e.g., "He is explaining this to convince her", "She is joking to lighten the mood")

    Do NOT count:
    - Pure descriptions of observable behavior or actions with no stated purpose
    (e.g., "He is talking to her", "She is sitting down", "He waves his hand")
    - Pure emotional states
    (e.g., "He feels nervous", "She is happy")
    - Pure beliefs or thoughts without a goal
    (e.g., "He thinks she is upset", "She believes he is lying")

    Important:
    - If a goal, purpose, or desired outcome is explicitly or implicitly stated, answer "yes".
    - If only actions, emotions, or beliefs are described without a goal, answer "no".

    Text:
    {text}
    """
)

# I think true charactertic would also include the feel for a single person.
# But if concatonated with emotion, i think it gives a good signal.
situation_characteristic_prompt = (
    """
    Your task is to determine whether the given text contains ANY description of the overall tone or atmosphere of the situation.

    Definition:
    Situation characteristics refer to the overall feel, tone, or atmosphere of the situation as a whole, rather than the state of a specific individual.

    A reference to situation tone includes:
    - Descriptions of the general atmosphere or mood of the scene
    (e.g., "The situation is tense", "It is a relaxed environment", "The interaction feels awkward")
    - Global interpretations of how the situation feels overall
    (e.g., "There is a sense of discomfort", "The mood is friendly", "The situation is formal")

    Do NOT count:
    - Emotional or mental states of a specific individual
    (e.g., "He feels nervous", "She is happy")
    - Pure descriptions of actions or events
    (e.g., "They are talking", "He walks into the room")
    - Intentions, beliefs, or goals
    (e.g., "He wants to leave", "She thinks he is wrong")

    Important:
    - The description must apply to the situation as a whole, not just one person.
    - If the text describes a shared or overall atmosphere, answer "yes".
    - If it only describes individuals or actions without describing the overall tone, answer "no".

    Text:
    {text}
    """
)

class_prompt = (
    """
    Your task is to determine whether the given text contains ANY reference to the type of situation the participant is in.

    Definition:
    Situation type refers to how the situation is categorized or understood (e.g., what kind of social scenario it is), from the perspective of the participant, including assumptions about roles, relationships, or context that define the situation.

    A reference to situation type includes:
    - Explicit labels or categories of the situation
    (e.g., "This is a job interview", "They are on a date", "This is a meeting", "It is a classroom setting")
    - Descriptions that clearly identify the kind of interaction or social context
    (e.g., "They are having a formal discussion", "This seems like a negotiation", "It appears to be a confrontation")
    - Assumptions about roles, relationships, or context that define the situation
    (e.g., "They are strangers", "He is their boss", "She knows him from a previous meeting")

    Do NOT count:
    - Descriptions of tone or atmosphere
    (e.g., "The situation is tense", "It feels relaxed")
    - Pure descriptions of actions or events
    (e.g., "They are talking", "He enters the room")
    - Mental states, beliefs, or intentions about individuals
    (e.g., "He thinks she is wrong", "She wants to leave")

    Important:
    - The text must indicate what kind of situation or interaction this is, not just what is happening.
    - If the situation is categorized or identified (explicitly or implicitly), answer "yes".
    - If only actions, tone, or individual states are described, answer "no".

    Text:
    {text}
    """
)

scripts_prompt = (
    """
    Your task is to determine whether the given text contains ANY reference to social scripts.

    Definition:
    Social scripts are generalized, typical patterns or “how-to” guides for how social interactions usually unfold. 
    They describe expected sequences of behavior associated with a type of situation, rather than specific actions in this instance.

    A reference to social scripts includes:
    - Descriptions of typical or expected sequences of interaction
    (e.g., "When people meet, they usually greet each other and introduce themselves")
    - Generalized statements about how people behave in certain situations
    (e.g., "In a job interview, you usually answer questions and present yourself professionally")
    - Implicit references to shared social norms or rituals
    (e.g., "They are following the usual greeting routine", "This follows a typical conversation pattern")

    Do NOT count:
    - Descriptions of specific actions happening in this instance
    (e.g., "He shakes her hand", "They start talking")
    - Intentions, beliefs, or emotions of individuals
    (e.g., "He wants to impress her", "She thinks he is rude")
    - Situation type labels without describing a typical pattern
    (e.g., "This is a meeting", "They are on a date")

    Important:
    - The text must refer to what typically or usually happens, not just what is happening now.
    - If the text describes a general pattern, expectation, or social norm, answer "yes".
    - If it only describes a specific instance without generalization, answer "no".

    Text:
    {text}
    """
)
narrative_fields = [
    'intention_description',
    'intention_description_confidence',
    'intention_explanation',
    'intention_explanation_confidence',
    'intention_intensity',
    'counterfactual_explanation',
]

llm_fields = ["mentions_emotion", "mentions_cues", "mentions_belief", "mentions_intention", "mentions_situation_characteristic", "mentions_class_prompt", "mentions_scripts_prompt"]
narrative_fields.extend(llm_fields)


In [ ]:
# Run LLM parsing on df_last_only and add binary flags for each classifier
# Result will be stored in `df_last_only_LLM_parse`.
df_form_B =  pd.read_csv('df_form_B_with_emotion.csv')
df_form_B = df_form_B.drop(columns = ['mentions_emotion', 'mentions_cues', 'mentions_belief', 'mentions_intention'], errors='ignore')
df_last_only_LLM_parse = df_form_B.copy()

# Sequentially add columns using the defined prompt templates
# Note: this calls OpenAI; ensure your api.key file exists and you have quota.

df_last_only_LLM_parse = add_emotion_flag(
    df_last_only_LLM_parse,
    user_prompt_template=emotion_prompt_v2,
    output_col="mentions_emotion",
    model="chat",
)

df_last_only_LLM_parse = add_emotion_flag(
    df_last_only_LLM_parse,
    user_prompt_template=cues_prompt_v2,
    output_col="mentions_cues",
    model="chat",
)

df_last_only_LLM_parse = add_emotion_flag(
    df_last_only_LLM_parse,
    user_prompt_template=belief_prompt_v2,
    output_col="mentions_belief",
    model="chat",
)

df_last_only_LLM_parse = add_emotion_flag(
    df_last_only_LLM_parse,
    user_prompt_template=intention_prompt_v2,
    output_col="mentions_intention",
    model="chat",
)

# df_last_only_LLM_parse = add_emotion_flag(
#     df_last_only_LLM_parse,
#     user_prompt_template=situation_characteristic_prompt,
#     output_col="mentions_situation_characteristic",
#     model="chat",
# )

# df_last_only_LLM_parse = add_emotion_flag(
#     df_last_only_LLM_parse,
#     user_prompt_template=class_prompt,
#     output_col="mentions_class_prompt",
#     model="chat",
# )

# df_last_only_LLM_parse = add_emotion_flag(
#     df_last_only_LLM_parse,
#     user_prompt_template=scripts_prompt,
#     output_col="mentions_scripts_prompt",
#     model="chat",
# )

# print(f"LLM parsing complete. Rows processed: {len(df_last_only_LLM_parse)}")

In [ ]:
llm_fields = ["intention_description", "intention_explanation", "mentions_emotion", "mentions_cues", "mentions_belief", "mentions_intention"]
table_df_final_LLM_parse = df_last_only_LLM_parse[llm_fields].copy()
for col in table_df_final_LLM_parse.columns:
    table_df_final_LLM_parse[col] = table_df_final_LLM_parse[col].fillna('').astype(str)

print('Displaying interactive table for Form Option A narrative fields:')
show(
    table_df_final_LLM_parse,
    scrollX=True,
    scrollY='320px',
    paging=False,
    order=[],
    columnDefs=[{'targets': '_all', 'className': 'dt-head-left dt-body-left'}]
)

In [ ]:
# Save enriched Form A and Form B dataframes to CSV
output_final = Path("df_form_B_LLM_parse.csv")

df_last_only_LLM_parse.to_csv(output_final, index=False)

print(f"Saved Form A dataframe to: {output_final.resolve()}")
print(f"Rows saved: {len(df_last_only_LLM_parse)}")

#### Stats

In [ ]:
df_final_LLM_parse = pd.read_csv("df_model_LLM_parse.csv")

print(f"Loaded df_last_only_LLM_parse.csv into df_final_LLM_parse: {df_final_LLM_parse.shape[0]} rows x {df_final_LLM_parse.shape[1]} columns")

In [ ]:
for column in ["mentions_emotion", "mentions_cues", "mentions_belief", "mentions_intention"]:
    # Count occurrences of True 
    count = df_final_LLM_parse[column].sum()
    total_count = len(df_final_LLM_parse)
    percentage = (count / total_count * 100) if total_count > 0 else 0

    print(f"{column}: {count} out of {total_count} ({percentage:.2f}%)")

In [ ]:
from itertools import combinations

stats_columns = [
    "mentions_emotion",
    "mentions_cues",
    "mentions_belief",
    "mentions_intention",
]

def _as_bool_series(series):
    return series.astype(str).str.lower().isin(["true", "1", "yes"])

def print_combination_stats(frame, frame_name, columns):
    bool_frame = frame[columns].apply(_as_bool_series)
    total_count = len(bool_frame)
    
    print(f"\n{frame_name} combination stats:")
    
    for size in range(1, len(columns) + 1):
        for combo in combinations(columns, size):
            mask = bool_frame.loc[:, combo].all(axis=1)
            count = int(mask.sum())
            percentage = (count / total_count * 100) if total_count > 0 else 0
            combo_name = " + ".join(combo)
            print(f"{combo_name}: {count} out of {total_count} ({percentage:.2f}%)")
    
    print(f"\n{frame_name} intention-focused pair stats:")
    for other_column in ["mentions_emotion", "mentions_cues", "mentions_belief"]:
        combo = ["mentions_intention", other_column]
        mask = bool_frame.loc[:, combo].all(axis=1)
        count = int(mask.sum())
        percentage = (count / total_count * 100) if total_count > 0 else 0
        combo_name = " + ".join(combo)
        print(f"{combo_name}: {count} out of {total_count} ({percentage:.2f}%)")

print_combination_stats(df_final_LLM_parse, "Dataset", stats_columns)

In [ ]:
import pandas as pd


def _build_interval_table(frame):
    interval_table = frame[["video_id", "timestamp_start", "timestamp_end"]].copy()
    interval_table["timestamp_start"] = pd.to_numeric(interval_table["timestamp_start"], errors="coerce")
    interval_table["timestamp_end"] = pd.to_numeric(interval_table["timestamp_end"], errors="coerce")
    interval_table = interval_table.dropna(subset=["video_id", "timestamp_start", "timestamp_end"])
    interval_table = interval_table[interval_table["timestamp_end"] >= interval_table["timestamp_start"]].copy()
    return interval_table


def _coverage_by_min_overlap(intervals, max_overlap=6):
    """Return coverage seconds where at least k intervals overlap, for k=1..max_overlap."""
    coverage = {k: 0.0 for k in range(1, max_overlap + 1)}
    if not intervals:
        return coverage

    events = {}
    for start, end in intervals:
        events[start] = events.get(start, 0) + 1
        events[end] = events.get(end, 0) - 1

    times = sorted(events.keys())
    active_count = 0
    prev_time = None

    for current_time in times:
        if prev_time is not None and current_time > prev_time:
            segment_duration = current_time - prev_time
            for k in range(1, max_overlap + 1):
                if active_count >= k:
                    coverage[k] += segment_duration

        active_count += events[current_time]
        prev_time = current_time

    return coverage


def _duration_stats(intervals):
    durations = [end - start for start, end in intervals]
    if not durations:
        return 0.0, 0.0, 0.0

    duration_series = pd.Series(durations, dtype="float64")
    avg_seconds = float(duration_series.mean())
    std_seconds = float(duration_series.std(ddof=0))
    var_seconds = float(duration_series.var(ddof=0))
    return avg_seconds, std_seconds, var_seconds


def calculate_video_overlap_stats(frame, frame_name, video_length_seconds=30):
    interval_table = _build_interval_table(frame)
    rows = []
    all_intervals = []
    per_video_coverages = []

    for video_id, group in interval_table.groupby("video_id", sort=True):
        intervals = [
            (row["timestamp_start"], row["timestamp_end"])
            for _, row in group[["timestamp_start", "timestamp_end"]].iterrows()
        ]
        all_intervals.extend(intervals)

        coverage = _coverage_by_min_overlap(intervals, max_overlap=6)
        per_video_coverages.append(coverage)
        avg_seconds, std_seconds, var_seconds = _duration_stats(intervals)

        row = {
            "video_id": video_id,
            "annotation_count": len(intervals),
            "avg_timestamp_duration_seconds": avg_seconds,
            "std_timestamp_duration_seconds": std_seconds,
            "var_timestamp_duration_seconds": var_seconds,
            "timestamp_duration_mean_plus_minus_seconds": f"{avg_seconds:.3f} +/- {std_seconds:.3f}",
            "overlapping_coverage_seconds": coverage[2],
            "overlapping_coverage_percent": (coverage[2] / video_length_seconds * 100) if video_length_seconds > 0 else 0.0,
        }

        for k in range(2, 7):
            row[f"coverage_seconds_at_least_{k}"] = coverage[k]
            row[f"coverage_percent_at_least_{k}"] = (coverage[k] / video_length_seconds * 100) if video_length_seconds > 0 else 0.0

        rows.append(row)

    result = pd.DataFrame(rows).sort_values("video_id").reset_index(drop=True)

    # Combined coverage should aggregate per-video coverage and normalize by total available time.
    # This avoids mixing different videos on the same 0..video_length timeline.
    video_count = len(per_video_coverages)
    total_available_seconds = video_count * video_length_seconds if video_length_seconds > 0 else 0
    combined_coverage = {k: sum(cov[k] for cov in per_video_coverages) for k in range(1, 7)} if per_video_coverages else {k: 0.0 for k in range(1, 7)}

    overall_avg_seconds, overall_std_seconds, overall_var_seconds = _duration_stats(all_intervals)
    overall_row = {
        "scope": "combined_all_videos",
        "video_count": video_count,
        "annotation_count": len(all_intervals),
        "avg_timestamp_duration_seconds": overall_avg_seconds,
        "std_timestamp_duration_seconds": overall_std_seconds,
        "var_timestamp_duration_seconds": overall_var_seconds,
        "timestamp_duration_mean_plus_minus_seconds": f"{overall_avg_seconds:.3f} +/- {overall_std_seconds:.3f}",
        "overlapping_coverage_seconds": combined_coverage[2],
        "overlapping_coverage_percent": (combined_coverage[2] / total_available_seconds * 100) if total_available_seconds > 0 else 0.0,
    }

    for k in range(2, 7):
        overall_row[f"coverage_seconds_at_least_{k}"] = combined_coverage[k]
        overall_row[f"coverage_percent_at_least_{k}"] = (combined_coverage[k] / total_available_seconds * 100) if total_available_seconds > 0 else 0.0

    overall_summary = pd.DataFrame([overall_row])

    print(f"\n{frame_name} timestamp overlap coverage (per unique video_id):")
    display(result)
    print(f"\n{frame_name} overall summary across all videos:")
    display(overall_summary)
    print(f"\n{frame_name} duration variability (seconds):")
    print(f"Overall mean +/- std: {overall_avg_seconds:.3f} +/- {overall_std_seconds:.3f} seconds")

    return result, overall_summary


overlap_stats_combined, overlap_summary_combined = calculate_video_overlap_stats(df_final_LLM_parse, "Dataset")

In [ ]:
summary = overlap_summary_combined.iloc[0]

video_count = int(summary["video_count"])
base_seconds = video_count * 30

print(f"video_count = {video_count}")
print(f"denominator (video_count * 30s) = {base_seconds}")
print(f"overlap>=2 seconds = {summary['coverage_seconds_at_least_2']:.6f}")
print(f"overlap>=3 seconds = {summary['coverage_seconds_at_least_3']:.6f}")
print(f"overlap>=4 seconds = {summary['coverage_seconds_at_least_4']:.6f}")
print(f"overlap>=5 seconds = {summary['coverage_seconds_at_least_5']:.6f}")
print(f"overlap>=6 seconds = {summary['coverage_seconds_at_least_6']:.6f}")

# Recompute percentages directly from seconds to validate the summary row.
for k in range(2, 7):
    sec = float(summary[f"coverage_seconds_at_least_{k}"])
    pct_recomputed = (sec / base_seconds) * 100 if base_seconds else 0.0
    pct_reported = float(summary[f"coverage_percent_at_least_{k}"])
    diff = abs(pct_recomputed - pct_reported)
    print(
        f"k={k}: recomputed={pct_recomputed:.6f} reported={pct_reported:.6f} diff={diff:.12f}"
    )

In [ ]:
import pandas as pd
from pathlib import Path

MIN_SEGMENT_SECONDS = 2.0

# Use df_final_LLM_parse as the source data
source_frame = df_final_LLM_parse.copy()

interval_table = source_frame[["video_id", "timestamp_start", "timestamp_end"]].copy()
interval_table["timestamp_start"] = pd.to_numeric(interval_table["timestamp_start"], errors="coerce")
interval_table["timestamp_end"] = pd.to_numeric(interval_table["timestamp_end"], errors="coerce")
interval_table = interval_table.dropna(subset=["video_id", "timestamp_start", "timestamp_end"])
interval_table = interval_table[interval_table["timestamp_end"] >= interval_table["timestamp_start"]].copy()


def build_concurrency_timeline(frame):
    rows = []

    for video_id, group in frame.groupby("video_id", sort=True):
        events = {}
        for _, row in group.iterrows():
            start = float(row["timestamp_start"])
            end = float(row["timestamp_end"])
            events[start] = events.get(start, 0) + 1
            events[end] = events.get(end, 0) - 1

        if not events:
            continue

        times = sorted(events.keys())
        active = 0

        for i in range(len(times) - 1):
            t = times[i]
            next_t = times[i + 1]
            active += events[t]
            duration = next_t - t

            if duration > 0 and active > 0:
                rows.append(
                    {
                        "video_id": video_id,
                        "segment_start": t,
                        "segment_end": next_t,
                        "segment_duration_seconds": duration,
                        "concurrent_annotations": int(active),
                        "overlapping_other_annotations": int(active - 1),
                    }
                )

    return pd.DataFrame(rows)


concurrency_timeline = build_concurrency_timeline(interval_table)

if concurrency_timeline.empty:
    print("No valid timestamp intervals found.")
else:
    # Keep only segments longer than the required threshold.
    long_segments = concurrency_timeline[
        concurrency_timeline["segment_duration_seconds"] > MIN_SEGMENT_SECONDS
    ].copy()

    if long_segments.empty:
        print(f"No concurrency segments longer than {MIN_SEGMENT_SECONDS} seconds were found.")
    else:
        # Per-video aggregation: proportion of long-segment time at each overlap level.
        per_video = (
            long_segments
            .groupby(["video_id", "concurrent_annotations", "overlapping_other_annotations"], as_index=False)["segment_duration_seconds"]
            .sum()
            .sort_values(["video_id", "concurrent_annotations"])
        )

        per_video_total = (
            per_video.groupby("video_id", as_index=False)["segment_duration_seconds"]
            .sum()
            .rename(columns={"segment_duration_seconds": "total_long_segment_seconds"})
        )
        per_video = per_video.merge(per_video_total, on="video_id", how="left")
        per_video["proportion_of_long_segments"] = (
            per_video["segment_duration_seconds"] / per_video["total_long_segment_seconds"]
        )

        # Dataset-wide aggregation: full-picture proportion at each overlap level.
        dataset_summary = (
            long_segments
            .groupby(["concurrent_annotations", "overlapping_other_annotations"], as_index=False)["segment_duration_seconds"]
            .sum()
            .sort_values("concurrent_annotations")
        )
        dataset_total = dataset_summary["segment_duration_seconds"].sum()
        dataset_summary["proportion_of_long_segments"] = (
            dataset_summary["segment_duration_seconds"] / dataset_total
        )

        # Helpful diagnostics for your research reporting.
        per_video_peak = (
            long_segments.groupby("video_id", as_index=False)["concurrent_annotations"]
            .max()
            .rename(columns={"concurrent_annotations": "max_annotators_on_same_moment_over_2s"})
        )

        print(f"Long overlap segments only: duration > {MIN_SEGMENT_SECONDS} seconds")
        print("\nMoment-level long segments (first 40):")
        display(long_segments.head(40))

        print("\nPer-video overlap profile (long segments only):")
        display(per_video)

        print("\nPer-video peak overlap (long segments only):")
        display(per_video_peak)

        print("\nDataset-wide overlap profile (long segments only):")
        display(dataset_summary)

        print("\nInterpretation:")
        print("- concurrent_annotations = N means the moment is annotated by N people at once.")
        print("- overlapping_other_annotations = N - 1 is how many other annotators overlap a given annotator's moment.")
        print("- proportion_of_long_segments is a fraction; multiply by 100 for percent.")